In [1]:
import pandas as pd
import numpy as np

column_names = [ "id", "diagnosis", "radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean", "compactness_mean", "concavity_mean", "concave_points_mean", "symmetry_mean", "fractal_dimension_mean", "radius_se", "texture_se", "perimeter_se", "area_se", "smoothness_se", "compactness_se", "concavity_se", "concave_points_se", "symmetry_se", "fractal_dimension_se", "radius_worst", "texture_worst", "perimeter_worst", "area_worst", "smoothness_worst", "compactness_worst", "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst" ]

cancer_df = pd.read_csv("wdbc.data", header =None , names = column_names)
print("Database shape:", cancer_df.shape)
cancer_df.head()

Database shape: (569, 32)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [2]:
print("Number of rows:", cancer_df.shape[0])
print("Number of columns:", cancer_df.shape[1])

print("\nTarget distribution:")
print(cancer_df["diagnosis"].value_counts())

print("\nMissing values:")
print(cancer_df.isnull().sum().sum())



Number of rows: 569
Number of columns: 32

Target distribution:
diagnosis
B    357
M    212
Name: count, dtype: int64

Missing values:
0


In [3]:
""" I first checked the dataset dimensions, target distribution and missing values. Since the dataset had no missing values, I didn't perform unnecessary imputation. I removed the ID because it doesn't carry predictive information and encoded the diagnosis into a binary target. """ 

" I first checked the dataset dimensions, target distribution and missing values. Since the dataset had no missing values, I didn't perform unnecessary imputation. I removed the ID because it doesn't carry predictive information and encoded the diagnosis into a binary target. "

In [4]:
# The ID is only an identifier, so it will not be used for prediction.
model_data = cancer_df.drop(columns=["id"]).copy()

# Convert the diagnosis into a binary numerical target.
# M = 1 (malignant), B = 0 (benign)
model_data["diagnosis"] = model_data["diagnosis"].map({
    "M": 1,
    "B": 0
})

print("Data shape after removing ID:", model_data.shape)
print("\nTarget values:")
print(model_data["diagnosis"].value_counts())

Data shape after removing ID: (569, 31)

Target values:
diagnosis
0    357
1    212
Name: count, dtype: int64


In [5]:
# Separate input features and target variable
X = model_data.drop(columns=["diagnosis"])
y = model_data["diagnosis"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


Feature matrix shape: (569, 30)
Target shape: (569,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


Training samples: 455
Testing samples: 114

Training target distribution:
diagnosis
0    285
1    170
Name: count, dtype: int64

Testing target distribution:
diagnosis
0    72
1    42
Name: count, dtype: int64


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled testing data shape:", X_test_scaled.shape)


Scaled training data shape: (455, 30)
Scaled testing data shape: (114, 30)


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier


In [9]:
# Creating the five classification models required for the assignment

logistic_model = LogisticRegression(max_iter=1000, random_state=42)

tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

knn_model = KNeighborsClassifier(n_neighbors=7)

naive_bayes_model = GaussianNB()

forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

print("All five models have been initialized.")


All five models have been initialized.


In [10]:
# Train Logistic Regression on scaled features
logistic_model.fit(X_train_scaled, y_train)

# Train Decision Tree on original features
tree_model.fit(X_train, y_train)

# Train KNN on scaled features
knn_model.fit(X_train_scaled, y_train)

# Train Gaussian Naive Bayes on original features
naive_bayes_model.fit(X_train, y_train)

# Train Random Forest on original features
forest_model.fit(X_train, y_train)

print("All five models trained successfully.")

All five models trained successfully.


In [11]:
# Class predictions
logistic_pred = logistic_model.predict(X_test_scaled)
tree_pred = tree_model.predict(X_test)
knn_pred = knn_model.predict(X_test_scaled)
naive_bayes_pred = naive_bayes_model.predict(X_test)
forest_pred = forest_model.predict(X_test)

# Probability estimates for the positive class (M = 1)
logistic_prob = logistic_model.predict_proba(X_test_scaled)[:, 1]
tree_prob = tree_model.predict_proba(X_test)[:, 1]
knn_prob = knn_model.predict_proba(X_test_scaled)[:, 1]
naive_bayes_prob = naive_bayes_model.predict_proba(X_test)[:, 1]
forest_prob = forest_model.predict_proba(X_test)[:, 1]

print("Predictions generated for all five models.")

Predictions generated for all five models.


In [12]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

def evaluate_model(actual, predicted, probabilities):
    return {
        "Accuracy": accuracy_score(actual, predicted),
        "AUC": roc_auc_score(actual, probabilities),
        "Precision": precision_score(actual, predicted),
        "Recall": recall_score(actual, predicted),
        "F1 Score": f1_score(actual, predicted),
        "MCC": matthews_corrcoef(actual, predicted)
    }

In [13]:
results = {}

results["Logistic Regression"] = evaluate_model(
    y_test, logistic_pred, logistic_prob
)

results["Decision Tree"] = evaluate_model(
    y_test, tree_pred, tree_prob
)

results["KNN"] = evaluate_model(
    y_test, knn_pred, knn_prob
)

results["Naive Bayes"] = evaluate_model(
    y_test, naive_bayes_pred, naive_bayes_prob
)

results["Random Forest"] = evaluate_model(
    y_test, forest_pred, forest_prob
)

results_df = pd.DataFrame(results).T

results_df = results_df.round(4)

results_df

,Accuracy,AUC,Precision,Recall,F1 Score,MCC
Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
Decision Tree,0.9211,0.9448,0.9459,0.8333,0.8861,0.8299
KNN,0.9561,0.9825,0.9744,0.9048,0.9383,0.9058
Naive Bayes,0.9386,0.9934,1.0000,0.8333,0.9091,0.8715
Random Forest,0.9649,0.9940,1.0000,0.9048,0.9500,0.9258


In [14]:
winner_by_f1 = results_df["F1 Score"].idxmax()
winner_by_auc = results_df["AUC"].idxmax()
winner_by_accuracy = results_df["Accuracy"].idxmax()

print("Best F1 Score:", winner_by_f1)
print("Best AUC:", winner_by_auc)
print("Best Accuracy:", winner_by_accuracy)

Best F1 Score: Logistic Regression
Best AUC: Logistic Regression
Best Accuracy: Logistic Regression


In [15]:
from sklearn.metrics import confusion_matrix, classification_report

best_model_name = results_df["F1 Score"].idxmax()

if best_model_name == "Logistic Regression":
    best_predictions = logistic_pred
elif best_model_name == "Decision Tree":
    best_predictions = tree_pred
elif best_model_name == "KNN":
    best_predictions = knn_pred
elif best_model_name == "Naive Bayes":
    best_predictions = naive_bayes_pred
else:
    best_predictions = forest_pred

print("Best model based on F1 Score:", best_model_name)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    best_predictions,
    target_names=["Benign", "Malignant"]
))

Best model based on F1 Score: Logistic Regression

Confusion Matrix:
[[71  1]
 [ 3 39]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        72
   Malignant       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [16]:
import joblib
import os

os.makedirs("saved_models", exist_ok=True)

In [17]:
joblib.dump(logistic_model, "saved_models/logistic_regression.pkl")
joblib.dump(tree_model, "saved_models/decision_tree.pkl")
joblib.dump(knn_model, "saved_models/knn.pkl")
joblib.dump(naive_bayes_model, "saved_models/naive_bayes.pkl")
joblib.dump(forest_model, "saved_models/random_forest.pkl")

# Save the scaler because Logistic Regression and KNN need scaled inputs
joblib.dump(scaler, "saved_models/scaler.pkl")

# Save the feature names so the Streamlit app can maintain the same order
joblib.dump(list(X.columns), "saved_models/feature_names.pkl")

print("All models and preprocessing objects saved successfully.")

All models and preprocessing objects saved successfully.


In [18]:
test_data = X_test.copy()
test_data["diagnosis"] = y_test.values

test_data.to_csv("test_data.csv", index=False)
print("Test data saved successfully.")
print("Test data shape:", test_data.shape)

Test data saved successfully.
Test data shape: (114, 31)


In [19]:
results_df.to_csv("model_results.csv")

print("Model comparison results saved.")

Model comparison results saved.


In [20]:
best_confusion_matrix = confusion_matrix(y_test, best_predictions)

print(best_confusion_matrix)

[[71  1]
 [ 3 39]]


In [21]:
"""OBSERVATION:
Logistic Regression
Logistic Regression gave the best overall performance in this experiment. It achieved 96.49% accuracy, the highest AUC of 0.9960, and the highest F1 score of 0.9512. Its recall of 92.86% also indicates that it was able to correctly identify most of the malignant cases in the test set.
Decision Tree
The Decision Tree produced the lowest performance among the five models, with 92.11% accuracy and an F1 score of 0.8861. Its recall of 83.33% indicates that it missed more malignant cases than Logistic Regression and KNN.
KNN
KNN performed strongly, achieving 95.61% accuracy and an F1 score of 0.9383. Its results were close to Logistic Regression, although its AUC and recall were slightly lower.
Naive Bayes
Naive Bayes achieved perfect precision of 1.0000, meaning none of its malignant predictions were incorrect in the test set. However, its recall was 83.33%, so it failed to identify some malignant cases.
Random Forest
Random Forest matched Logistic Regression in accuracy at 96.49% and achieved perfect precision. It also produced the highest MCC score of 0.9258. However, its recall and F1 score were marginally lower than Logistic Regression.
Overall Winner
Logistic Regression can be considered the overall winner for this dataset because it achieved the highest AUC, recall and F1 score while also matching Random Forest for the highest accuracy. Random Forest was a very close second and had the highest MCC."""


'OBSERVATION:\nLogistic Regression\nLogistic Regression gave the best overall performance in this experiment. It achieved 96.49% accuracy, the highest AUC of 0.9960, and the highest F1 score of 0.9512. Its recall of 92.86% also indicates that it was able to correctly identify most of the malignant cases in the test set.\nDecision Tree\nThe Decision Tree produced the lowest performance among the five models, with 92.11% accuracy and an F1 score of 0.8861. Its recall of 83.33% indicates that it missed more malignant cases than Logistic Regression and KNN.\nKNN\nKNN performed strongly, achieving 95.61% accuracy and an F1 score of 0.9383. Its results were close to Logistic Regression, although its AUC and recall were slightly lower.\nNaive Bayes\nNaive Bayes achieved perfect precision of 1.0000, meaning none of its malignant predictions were incorrect in the test set. However, its recall was 83.33%, so it failed to identify some malignant cases.\nRandom Forest\nRandom Forest matched Logist